# 02. 온라인 전처리 (블록 ①~⑤ → 3D_raw_online.xlsx)

| 항목 | 내용 |
|---|---|
| 입력 | 프레임 정렬된 상단·정면 영상, YOLO 가중치(top/front .pt), 보정 NPZ (01에서 생성) |
| 출력 | `<결과 폴더>/3D_raw_online.xlsx`, 주석 영상 `analyzed_output.mp4`(선택), `verify_unification.png`, `config_used.yaml` |
| GPU | **필요** — 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` |
| 바꿀 것 | 1번 셀의 `CONFIG_PATH` 하나 |

프레임마다 아래 블록을 순서대로 처리한다. 블록이 프레임 반복문 안에서 연속으로 돌기 때문에 셀을 블록별로 나누지 않고,
코드 파일을 블록별로 나눴다 (`gtrack/online/`).

| 블록 | 내용 | 코드 |
|---|---|---|
| ① 물체 검출 | YOLO 경계상자 중심 = 대표 픽셀 | `b01_detection.py` |
| ◇ 두 시점 검출 성립? | 아니면 TOP/FRONT/BOTH_MISSING으로 ④에 전달 | `b01_detection.pairing_reason` |
| ② 광선 생성 · 굴절 | 벡터형 Snell, 굴절 ON/OFF 동시 | `b02_rays.py` → `gtrack/geometry.py` |
| ③ 최근접점 삼각측량 | Candidate_ON/OFF, Ray_Gap_cm | `b03_triangulation.py` |
| ④ 후보 수용 · 궤적 확정 | 광선 간 거리(≤7 cm) → 변위 → Mahalanobis → 7-of-10 | `b04_tracker.py` |
| ⑤ 그리퍼 중심 좌표 변환 | Meas_X/Y/Z_cm | `b05_motor_frame.py` |

In [ ]:
#@title 0. 환경 준비 (저장소 받기 · 라이브러리 설치 · Drive 연결)
REPO_URL = "https://github.com/YOUR_ID/underwater-gripper-tracking.git"  #@param {type:"string"}

import os, sys, subprocess, warnings
REPO_DIR = '/content/' + REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=False)
%pip install -q -r {REPO_DIR}/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

sys.path.insert(0, REPO_DIR)
import pandas as pd
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
print('준비 완료:', REPO_DIR)

In [ ]:
#@title 1. 설정 파일 지정 · 점검
CONFIG_PATH = "/content/drive/MyDrive/8월연구/config.yaml"  #@param {type:"string"}

import shutil
from gtrack import settings

if not os.path.exists(CONFIG_PATH):
    os.makedirs(os.path.dirname(CONFIG_PATH), exist_ok=True)
    shutil.copy(os.path.join(REPO_DIR, 'config.yaml'), CONFIG_PATH)
    raise RuntimeError(
        f'설정 파일이 없어 템플릿을 복사했습니다: {CONFIG_PATH}\n'
        '왼쪽 파일 패널에서 이 파일을 더블클릭해 [필수] 항목을 수정한 뒤 이 셀을 다시 실행하세요.')

settings.configure(CONFIG_PATH)
settings.print_summary('online')
settings.check('online')

In [ ]:
#@title 2. GPU 확인
import torch
if torch.cuda.is_available():
    print('GPU 사용:', torch.cuda.get_device_name(0))
else:
    print('⚠️ GPU가 없습니다. 런타임 → 런타임 유형 변경 → T4 GPU 선택 후 0번 셀부터 다시 실행하세요.')
    print('   (CPU로도 실행되지만 매우 느립니다)')

## 3. 짧게 시험 실행 (권장)

처음 300프레임만 처리해서 경로·모델·보정이 맞는지 먼저 확인한다. 결과 파일은 4번 전체 실행에서 덮어쓴다.
`verify_unification.png`에서 빨간 점 4개가 정면 화면의 바닥 마커 코너에 오는지 확인한다.

In [ ]:
#@title 3. 시험 실행 (처음 N프레임)
TEST_FRAMES = 300  #@param {type:"integer"}
from gtrack.online.pipeline import run_online
df_test = run_online(max_frames=TEST_FRAMES)

import cv2, matplotlib.pyplot as plt
if os.path.exists(settings.VERIFY_IMAGE_PATH):
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(cv2.imread(settings.VERIFY_IMAGE_PATH), cv2.COLOR_BGR2RGB))
    plt.title('verify_unification: red dots should sit on the floor marker corners'); plt.axis('off'); plt.show()

## 4. 전체 실행

영상 길이에 따라 수 분~수십 분 걸린다. 진행 상황은 10초 분량마다 출력된다.
주석 영상이 필요 없으면 config.yaml `video.save_annotated_video: false`로 두면 더 빠르다.

In [ ]:
#@title 4. 전체 실행
from gtrack.online.pipeline import run_online
df_raw = run_online()

In [ ]:
#@title 5. 결과 확인
import matplotlib.pyplot as plt
print(df_raw['Missing_Reason'].value_counts().to_string())
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
for ax, c in zip(axes, ['Meas_X', 'Meas_Y', 'Meas_Z']):
    ax.plot(df_raw['Time_sec'], df_raw[c], '.', ms=2)
    ax.set_ylabel(c + ' [cm]'); ax.grid(alpha=.3)
axes[-1].set_xlabel('Time [s]'); plt.tight_layout(); plt.show()
print('다음 단계: 03_postprocess.ipynb (같은 CONFIG_PATH 사용)')